 Ficha Técnica: Dataset Credit Approval (CRX)
Descripción General
Este dataset, proveniente del repositorio UCI Machine Learning, contiene solicitudes de tarjetas de crédito reales. Debido a la sensibilidad de la información bancaria, todos los nombres de las variables y sus valores originales fueron anonimizados y sustituidos por símbolos (A1, A2, etc.).

Es un dataset "sucio" en su origen, con una mezcla de datos categóricos (letras) y continuos (números), además de valores faltantes (identificados inicialmente como ?).



Columna,Tipo de Dato,Significado Probable
A1,Categórico,"Género / Estado Civil (a, b)"
A2,Continuo,Edad del solicitante
A3,Continuo,Deuda actual / Ratio de deuda
A4,Categórico,"Estado del cliente (Casado, Soltero, etc.)"
A5,Categórico,"Sector de empleo (Educación, Salud, etc.)"
A6,Categórico,Etnia / Nacionalidad
A7,Continuo,Años de experiencia laboral
A8,Categórico,¿Tiene deudas previas? (t/f)
A9,Categórico,¿Está empleado actualmente? (t/f)
A10,Continuo,Score crediticio / Número de cuentas
A11,Categórico,¿Tiene licencia de conducir / ID? (t/f)
A12,Categórico,Tipo de ciudadanía
A13,Continuo,Salario / Ingresos mensuales
A14,ELIMINADO,Código Postal (Zip Code)
A15,Continuo,Saldo en cuenta / Otros ingresos
Target,Binario,Veredicto: 1 (Aprobado) / 0 (Denegado)

In [1]:
# FASE 1: IMPORTACIONES
import pandas as pd
import numpy as np
from IPython.display import display

# FASE 2: CARGA DEL DATASET ORIGINAL (CRX)
ruta_crx = '/content/sample_data/crx.data'

# Definimos nombres de columnas (A1 a A16)
nombres_cols = [f'A{i}' for i in range(1, 16)] + ['Target']

# Cargamos indicando que el '?' es un valor nulo (NaN)
df_crudo = pd.read_csv(ruta_crx, header=None, names=nombres_cols, na_values='?')

print(f"✅ Dataset 'CRX' cargado. Filas: {df_crudo.shape[0]}, Columnas: {df_crudo.shape[1]}")
display(df_crudo.head(10))

✅ Dataset 'CRX' cargado. Filas: 690, Columnas: 16


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A14,A15,Target
0,b,30.83,0.000,u,g,w,v,1.250,t,t,1,f,g,202.0,0,+
1,a,58.67,4.460,u,g,q,h,3.040,t,t,6,f,g,43.0,560,+
2,a,24.50,0.500,u,g,q,h,1.500,t,f,0,f,g,280.0,824,+
3,b,27.83,1.540,u,g,w,v,3.750,t,t,5,t,g,100.0,3,+
4,b,20.17,5.625,u,g,w,v,1.710,t,f,0,f,s,120.0,0,+
5,b,32.08,4.000,u,g,m,v,2.500,t,f,0,t,g,360.0,0,+
6,b,33.17,1.040,u,g,r,h,6.500,t,f,0,t,g,164.0,31285,+
7,a,22.92,11.585,u,g,cc,v,0.040,t,f,0,f,g,80.0,1349,+
8,b,54.42,0.500,y,p,k,h,3.960,t,f,0,f,g,180.0,314,+
9,b,42.50,4.915,y,p,w,v,3.165,t,f,0,t,g,52.0,1442,+


In [5]:
# FASE 2.2: LIMPIEZA, DROPEO Y CODIFICACIÓN
df_limpio = df_crudo.copy()

# 1. ELIMINACIÓN DE COLUMNA A14 (Código Postal / Ruido)
df_limpio = df_limpio.drop(columns=['A14'])
print("✂️ Columna A14 eliminada por ser considerada ruido geográfico.")

# 2. IDENTIFICAR COLUMNAS RESTANTES
cols_num = df_limpio.select_dtypes(include=[np.number]).columns.tolist()
cols_cat = df_limpio.select_dtypes(include=['object']).columns.tolist()
cols_cat.remove('Target')

# 3. RELLENAR HUECOS (NaNs)
for col in cols_num:
    df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mean())
for col in cols_cat:
    df_limpio[col] = df_limpio[col].fillna(df_limpio[col].mode()[0])

# 4. CODIFICACIÓN DE LETRAS A NÚMEROS
for col in cols_cat:
    df_limpio[col], _ = pd.factorize(df_limpio[col])

# 5. TRANSFORMACIÓN DEL TARGET (+ -> 1, - -> 0)
df_limpio['Target'] = df_limpio['Target'].map({'+': 1, '-': 0})

print("✅ Limpieza e Imputación completada.")
display(df_limpio.head(5))

✂️ Columna A14 eliminada por ser considerada ruido geográfico.
✅ Limpieza e Imputación completada.


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A15,Target
0,0,30.83,0.000,0,0,0,0,1.25,0,0,1,0,0,0,1
1,1,58.67,4.460,0,0,1,1,3.04,0,0,6,0,0,560,1
2,1,24.50,0.500,0,0,1,1,1.50,0,1,0,0,0,824,1
3,0,27.83,1.540,0,0,0,0,3.75,0,0,5,1,0,3,1
4,0,20.17,5.625,0,0,0,0,1.71,0,1,0,0,1,0,1


In [6]:
# FASE 2.3: SEPARACIÓN DE X e y
# X: Atributos predictores (A1 a A13 + A15, sin el Target)
X = df_limpio.drop(columns=['Target'])

# y: Variable objetivo
y = df_limpio['Target']

print("🏗️ Separación de matrices finalizada.")
print(f"Dimensiones de X: {X.shape}")
print(f"Dimensiones de y: {y.shape}")

🏗️ Separación de matrices finalizada.
Dimensiones de X: (690, 14)
Dimensiones de y: (690,)


In [7]:
# FASE 2.4: MUESTRA DE DATOS SEPARADOS
print("📋 MATRIZ X (Características del Cliente - Sin A14):")
from IPython.display import display
display(X.head(5))

print("\n" + "="*60 + "\n")

print("🎯 VECTOR y (Decisión del Crédito: 0=No, 1=Sí):")
display(y.head(5))

# Resumen de Auditoría
print("-" * 60)
print(f"Variables en X: {list(X.columns)}")
print("Estado: LISTO PARA NORMALIZACIÓN (FASE 3)")

📋 MATRIZ X (Características del Cliente - Sin A14):


,A1,A2,A3,A4,A5,A6,A7,A8,A9,A10,A11,A12,A13,A15
0,0,30.83,0.000,0,0,0,0,1.25,0,0,1,0,0,0
1,1,58.67,4.460,0,0,1,1,3.04,0,0,6,0,0,560
2,1,24.50,0.500,0,0,1,1,1.50,0,1,0,0,0,824
3,0,27.83,1.540,0,0,0,0,3.75,0,0,5,1,0,3
4,0,20.17,5.625,0,0,0,0,1.71,0,1,0,0,1,0




🎯 VECTOR y (Decisión del Crédito: 0=No, 1=Sí):


,Target
0,1
1,1
2,1
3,1
4,1


------------------------------------------------------------
Variables en X: ['A1', 'A2', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'A9', 'A10', 'A11', 'A12', 'A13', 'A15']
Estado: LISTO PARA NORMALIZACIÓN (FASE 3)
